# Train v2 – reference-anchored MDD (Iqra'Eval 2025 data only)

**Settings (right panel):** Accelerator = *GPU T4 x2*, Internet = *On*, Persistence = *Files only*.
**Secrets (Add-ons → Secrets):** `HF_TOKEN` (read access is enough).

Run with **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 h and keeps `/kaggle/working`.
The script stops itself at ~11.3 h and saves `run/last/` so the next session can resume.

**To resume:** *Add Input* → your previous version of this notebook (its output appears under `/kaggle/input/<notebook-name>/`), set `PREV` below, run again.

In [ ]:
PREV = None   # e.g. "/kaggle/input/iqraeval-v2-train/run"  (previous session's output) – None for the first session
MAX_STEPS = 30000
BRANCH = "v2"


In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!git clone -q -b $BRANCH https://github.com/hBouanane/wav2vec2-mispronunciation-detector repo
!git clone -q --depth 1 https://github.com/Iqra-Eval/interspeech_IqraEval
!pip install -q jiwer "datasets>=3.0,<4" soundfile librosa
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
RUN = "/kaggle/working/run"
if PREV:
    !mkdir -p $RUN && cp -r $PREV/. $RUN/
resume = "--resume" if PREV else ""
!cd repo && accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 v2/train_v2.py     --out $RUN --mdd-eval ../interspeech_IqraEval/mdd_eval --max-steps $MAX_STEPS $resume 2>&1 | grep -v -i warn

In [ ]:
!cat $RUN/best.json 2>/dev/null | head -5; tail -3 $RUN/eval_log.jsonl 2>/dev/null
!du -sh $RUN/*